# Iterative Imputer — Step by Step

### How it works:
Instead of filling NaN with a fixed value (mean/median) or similar rows (KNN),
Iterative Imputer **trains a regression model** for each column with missing values.

It uses the other columns as features to **predict** the missing value.

It repeats this process multiple times (iterations) until the values **stop changing much**.

### Steps:
1. Fill all missing values with column mean (rough starting point)
2. For each column with missing values:
   - Remove its imputed value
   - Train a regression model using other columns
   - Predict the missing value
   - Fill it with the predicted value
3. Repeat step 2 until values converge (stop changing)

In [14]:
# Import required libraries
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression

In [15]:
# Load 50_Startups dataset — scale down values by 10000 for readability
# Randomly sample 5 rows to keep it simple for step-by-step demonstration
df = np.round(pd.read_csv('50_Startups.csv')[['R&D Spend', 'Administration', 'Marketing Spend', 'Profit']] / 10000)
np.random.seed(9)
df = df.sample(5)
df

,R&D Spend,Administration,Marketing Spend,Profit
21,8.0,15.0,30.0,11.0
37,4.0,5.0,20.0,9.0
2,15.0,10.0,41.0,19.0
14,12.0,16.0,26.0,13.0
44,2.0,15.0,3.0,7.0


In [16]:
# Drop the Profit column — we will only work with 3 feature columns
df = df.iloc[:, 0:-1]
df

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,4.0,5.0,20.0
2,15.0,10.0,41.0
14,12.0,16.0,26.0
44,2.0,15.0,3.0


In [17]:
# Manually introduce 3 missing values — one in each column
# Row 1, Col 0 → R&D Spend missing
# Row 3, Col 1 → Administration missing
# Last row, last col → Marketing Spend missing
df.iloc[1, 0] = np.nan
df.iloc[3, 1] = np.nan
df.iloc[-1, -1] = np.nan

df.head()

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,NaN,5.0,20.0
2,15.0,10.0,41.0
14,12.0,NaN,26.0
44,2.0,15.0,NaN


## Step 1 — Initial fill with column mean (0th iteration)

In [18]:
# Fill all NaN with column mean as a rough starting point
# This is just to get valid numbers so regression can run
df0 = pd.DataFrame()

df0['R&D Spend']       = df['R&D Spend'].fillna(df['R&D Spend'].mean())
df0['Administration']  = df['Administration'].fillna(df['Administration'].mean())
df0['Marketing Spend'] = df['Marketing Spend'].fillna(df['Marketing Spend'].mean())

print("0th iteration (mean filled):")
df0

0th iteration (mean filled):


,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,9.25,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


## Step 2 — 1st Iteration: predict each missing value using regression

In [19]:
# --- Fix R&D Spend (row 1) ---
# Remove the mean-filled value to bring back NaN
df1 = df0.copy()
df1.iloc[1, 0] = np.nan

# Train regression: use Administration + Marketing Spend to predict R&D Spend
# Use all rows EXCEPT row 1 (the missing one) for training
X = df1.iloc[[0, 2, 3, 4], 1:3]   # Administration, Marketing Spend
y = df1.iloc[[0, 2, 3, 4], 0]     # R&D Spend (target)

lr = LinearRegression()
lr.fit(X, y)

# Predict R&D Spend for row 1
predicted = lr.predict(df1.iloc[1, 1:].values.reshape(1, 2))
print(f"Predicted R&D Spend for row 1: {predicted[0]:.2f}")

df1.iloc[1, 0] = round(predicted[0], 2)
predicted 

Predicted R&D Spend for row 1: 23.14


c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([23.14158651])

In [20]:
# --- Fix Administration (row 3) ---
df1.iloc[3, 1] = np.nan

# Train regression: use R&D Spend + Marketing Spend to predict Administration
X = df1.iloc[[0, 1, 2, 4], [0, 2]]   # R&D Spend, Marketing Spend
y = df1.iloc[[0, 1, 2, 4], 1]        # Administration (target)

lr = LinearRegression()
lr.fit(X, y)

predicted = lr.predict(df1.iloc[3, [0, 2]].values.reshape(1, 2))
print(f"Predicted Administration for row 3: {predicted[0]:.2f}")

df1.iloc[3, 1] = round(predicted[0], 2)
df1

Predicted Administration for row 3: 11.06


c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,29.25


In [21]:
# --- Fix Marketing Spend (last row) ---
df1.iloc[4, -1] = np.nan

# Train regression: use R&D Spend + Administration to predict Marketing Spend
X = df1.iloc[0:4, 0:2]   # R&D Spend, Administration
y = df1.iloc[0:4, -1]    # Marketing Spend (target)

lr = LinearRegression()
lr.fit(X, y)

predicted = lr.predict(df1.iloc[4, 0:2].values.reshape(1, 2))
print(f"Predicted Marketing Spend for row 4: {predicted[0]:.2f}")

df1.iloc[4, -1] = round(predicted[0], 2)
df1

Predicted Marketing Spend for row 4: 31.56


c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,31.56


In [22]:
# How much did values change from iteration 0 to iteration 1?
# Large change = not converged yet, keep iterating
# Small/zero change = converged, stop
print("Change after 1st iteration:")
df1 - df0

Change after 1st iteration:


,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,13.89,0.00,0.00
2,0.00,0.00,0.00
14,0.00,-0.19,0.00
44,0.00,0.00,2.31


## Step 3 — 2nd Iteration: repeat the same process

In [23]:
# 2nd iteration — same process but now starting from df1 instead of df0
# Better starting values = better predictions
df2 = df1.copy()

# Fix R&D Spend
df2.iloc[1, 0] = np.nan
X = df2.iloc[[0, 2, 3, 4], 1:3]
y = df2.iloc[[0, 2, 3, 4], 0]
lr = LinearRegression()
lr.fit(X, y)
df2.iloc[1, 0] = round(lr.predict(df2.iloc[1, 1:].values.reshape(1, 2))[0], 2)

# Fix Administration
df2.iloc[3, 1] = np.nan
X = df2.iloc[[0, 1, 2, 4], [0, 2]]
y = df2.iloc[[0, 1, 2, 4], 1]
lr = LinearRegression()
lr.fit(X, y)
df2.iloc[3, 1] = round(lr.predict(df2.iloc[3, [0, 2]].values.reshape(1, 2))[0], 2)

# Fix Marketing Spend
df2.iloc[4, -1] = np.nan
X = df2.iloc[0:4, 0:2]
y = df2.iloc[0:4, -1]
lr = LinearRegression()
lr.fit(X, y)
df2.iloc[4, -1] = round(lr.predict(df2.iloc[4, 0:2].values.reshape(1, 2))[0], 2)

print("After 2nd iteration:")
print(df2)
print("\nChange from iteration 1 to 2:")
df2 - df1

After 2nd iteration:
    R&D Spend  Administration  Marketing Spend
21       8.00           15.00            30.00
37      23.79            5.00            20.00
2       15.00           10.00            41.00
14      12.00           11.22            26.00
44       2.00           15.00            38.94

Change from iteration 1 to 2:


c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,0.65,0.00,0.00
2,0.00,0.00,0.00
14,0.00,0.16,0.00
44,0.00,0.00,7.38


## Step 4 — 3rd Iteration

In [24]:
# 3rd iteration
df3 = df2.copy()

# Fix R&D Spend
df3.iloc[1, 0] = np.nan
X = df3.iloc[[0, 2, 3, 4], 1:3]
y = df3.iloc[[0, 2, 3, 4], 0]
lr = LinearRegression()
lr.fit(X, y)
df3.iloc[1, 0] = round(lr.predict(df3.iloc[1, 1:].values.reshape(1, 2))[0], 2)

# Fix Administration
df3.iloc[3, 1] = np.nan
X = df3.iloc[[0, 1, 2, 4], [0, 2]]
y = df3.iloc[[0, 1, 2, 4], 1]
lr = LinearRegression()
lr.fit(X, y)
df3.iloc[3, 1] = round(lr.predict(df3.iloc[3, [0, 2]].values.reshape(1, 2))[0], 2)

# Fix Marketing Spend
df3.iloc[4, -1] = np.nan
X = df3.iloc[0:4, 0:2]
y = df3.iloc[0:4, -1]
lr = LinearRegression()
lr.fit(X, y)
df3.iloc[4, -1] = round(lr.predict(df3.iloc[4, 0:2].values.reshape(1, 2))[0], 2)

print("After 3rd iteration:")
print(df3)
print("\nChange from iteration 2 to 3:")
df3 - df2

After 3rd iteration:
    R&D Spend  Administration  Marketing Spend
21       8.00           15.00            30.00
37      26.82            5.00            20.00
2       15.00           10.00            41.00
14      12.00           12.23            26.00
44       2.00           15.00            62.88

Change from iteration 2 to 3:


c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,3.03,0.00,0.00
2,0.00,0.00,0.00
14,0.00,1.01,0.00
44,0.00,0.00,23.94


## Step 5 — Use sklearn's IterativeImputer (does all of this automatically)

In [25]:
# sklearn's IterativeImputer does everything above automatically
# max_iter = how many iterations to run
# random_state = for reproducibility

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

imputer = IterativeImputer(max_iter=10, random_state=0)

result = np.round(imputer.fit_transform(df), 2)
result_df = pd.DataFrame(result, columns=df.columns)

print("Original with NaN:")
print(df)
print("\nAfter IterativeImputer:")
print(result_df)

Original with NaN:
    R&D Spend  Administration  Marketing Spend
21        8.0            15.0             30.0
37        NaN             5.0             20.0
2        15.0            10.0             41.0
14       12.0             NaN             26.0
44        2.0            15.0              NaN

After IterativeImputer:
   R&D Spend  Administration  Marketing Spend
0       8.00           15.00            30.00
1      10.71            5.00            20.00
2      15.00           10.00            41.00
3      12.00            6.33            26.00
4       2.00           15.00            12.99


## Summary

| Imputer | Strategy | Smart? |
|---|---|---|
| SimpleImputer | fixed mean/median value | No |
| KNNImputer | average of K similar rows | Partly |
| **IterativeImputer** | **regression model per column** | **Yes** |

**IterativeImputer is the most powerful** because:
- It uses ALL other columns to predict the missing value
- It refines predictions across multiple iterations
- Values converge to the most mathematically consistent result

**Downside:** Slowest of all imputers — especially on large datasets.